# 11 — From GPT-2 to DeepSeek (roadmap)

**Before:** notebooks **1–10** (GPT / llm.c path).

**This notebook:** why we teach **DeepSeek-V2** after GPT-2, not “V1”; map to `c/deepseek_v2/`.

**Learning objectives**

- Explain why DeepSeek-V2 follows GPT-2 in this curriculum.
- Compare GPT-2 block vs DeepSeek-V2 block (MLA + MoE).
- Estimate KV-cache size: MHA vs MLA.
- Locate the V2 C port under `c/deepseek_v2/`.

**Online course:** run cells **top-to-bottom**. Setup cell must print `data OK`.



**You finished notebooks 1–10** — you know tokens, causal attention, GPT blocks, training, and how that maps to Karpathy’s [llm.c](https://github.com/karpathy/llm.c).

**This notebook answers one question:**

> *Should we transform llm.c into **DeepSeek V1** first?*

**Short answer: no.** Use **DeepSeek-V2** as the first architecture step after GPT-2. Notebooks **12–16** teach V2; we port to C **one file at a time** under `c/deepseek_v2/`.


In [ ]:
# --- Setup: find repo root (llm-c-from-scratch or Cursor workbook) ---
import sys
from pathlib import Path


def find_llm_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "llmc" / "__init__.py").is_file():
            return base
        nested = base / "llm-c-from-scratch"
        if (nested / "llmc" / "__init__.py").is_file():
            return nested
    return Path.cwd()


ROOT = find_llm_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from llmc.notebook_utils import c_dir, checkpoint_path, data_path, v2_checkpoint_path

DATA = data_path(ROOT)
CHECKPOINT = checkpoint_path(ROOT)
C_DIR = c_dir(ROOT)
V2_CKPT = v2_checkpoint_path(ROOT)
print("ROOT", ROOT.resolve())
print("data", "OK" if DATA.is_file() else "missing")


## Should we port llm.c → “DeepSeek V1” first?

| Idea | Reality |
|------|--------|
| “V1” as a famous **architecture** (like V2 MLA + MoE) | **No** — there is no standard public “DeepSeek-V1 architecture” tutorial. |
| Early **DeepSeek** models (first releases) | Strong **dense** LMs, closer to **Llama/GPT** than to V2/V4. No separate lesson stack in this repo. |
| **DeepSeek-V2** (May 2024 paper) | First **named** jump: **MLA** (smaller KV cache) + **DeepSeekMoE** (sparse FFN). **This is step 2 after GPT-2.** |
| **DeepSeek-V4** | Another big redesign (mHC, CSA/HCA, …). **Step 3** — see `c/deepseek_v4_*` only after V2. |

### What “going to V1 first” would mean in practice

If you literally fork `train_gpt2.c` and only rename things, you still have **GPT-2 math** (LayerNorm, dense MLP, standard MHA). That is **not** a new lesson — you already did it in notebooks **1–10**.

If you mean “use the **oldest DeepSeek checkpoint**,” you still learn **dense transformer** mechanics first; the **new** ideas start at **V2**.

### Recommended ladder (this repo)

```text
llm.c / GPT-2     notebooks 1-10   +  vendor/llm.c/train_gpt2.c
DeepSeek-V2       notebooks 11-16  +  c/deepseek_v2/*.c (piece by piece)
DeepSeek-V4       notebook 18      +  c/swiglu.c, c/hash_moe.c (phased)
```

Read also: `docs/DEEPSEEK_VERSION_LADDER.md`.

## One GPT-2 block (what llm.c already implements)

In `train_gpt2.c`, one transformer layer roughly does:

1. **LayerNorm** → **causal self-attention** (Q,K,V from one linear, cache full K and V)
2. Residual add
3. **LayerNorm** → **dense MLP** (4× expansion, GELU) — *one* MLP for every token
4. Residual add

Our PyTorch mirror is `llmc.model.Block` (built in notebooks 5–6).

In [ ]:
from llmc.data import CharTokenizer, load_text
from llmc.model import GPT, GPTConfig

text = load_text(DATA)
tok = CharTokenizer.from_text(text)
cfg_gpt = GPTConfig.tiny(tok.vocab_size, block_size=32)
gpt = GPT(cfg_gpt)

# Print the *names* of submodules so you can grep the same ideas in train_gpt2.c
block0 = gpt.transformer.h[0]
print("GPT-2 block 0 children:")
for name, mod in block0.named_children():
    print(f"  {name:12} -> {mod.__class__.__name__}")

print("\nAttention uses ONE matrix c_attn: (C) -> (3C) for Q,K,V together.")
print("FFN is ONE dense MLP (c_fc, c_proj) — no router, no experts.")


## One DeepSeek-V2 block (what we add in notebooks 12–14)

Same *residual* pattern, **different internals**:

| Piece | GPT-2 / llm.c | DeepSeek-V2 |
|-------|----------------|-------------|
| Norm | LayerNorm | **RMSNorm** |
| Attention | MHA, cache full K,V per head | **MLA**, cache small **latent** `c_kv` |
| FFN | One dense MLP | **MoE**: router + top-k **experts** + **shared** expert |

Implementation: `llmc/deepseek_v2.py` (heavily commented — read it like a second textbook).

In [ ]:
from llmc.deepseek_v2 import DeepSeekV2, DeepSeekV2Config

cfg_ds = DeepSeekV2Config.tiny(tok.vocab_size, block_size=32)
ds = DeepSeekV2(cfg_ds)

b0 = ds.blocks[0]
print("DeepSeek-V2 block 0 children:")
for name, mod in b0.named_children():
    print(f"  {name:12} -> {mod.__class__.__name__}")

# Educational: compare KV cache *per token* (float32, one layer; multiply by n_layer for full model)
print("\nKV cache bytes per token (one layer):")
print("  MLA (V2):", ds.kv_cache_bytes_per_token() // cfg_ds.n_layer)
print("  MHA (GPT-2 style, hypothetical):", ds.mha_kv_cache_bytes_per_token() // cfg_ds.n_layer)
print("  ratio MHA/MLA:", ds.mha_kv_cache_bytes_per_token() / ds.kv_cache_bytes_per_token())


## Map: our notebooks ↔ llm.c ↔ our C port

Study **PyTorch first** (slow, clear), then read **commented C** (fast, matches llm.c style).

| Step | PyTorch notebook | llm.c (`train_gpt2.c`) | Our C (`c/deepseek_v2/`) |
|------|------------------|-------------------------|---------------------------|
| 11 | **This notebook** — roadmap | Overview / model struct | README |
| 12 | MLA | `attention_forward` | **`mla.c`** ✓ |
| 13 | DeepSeekMoE | *(no MoE in GPT-2)* | `moe.c` ✓ |
| 14 | Full V2 model | layer loop | `block.c` ✓ |
| 15 | Training | main train loop | `train_v2_tiny.c` ✓ |
| 16 | Sampling | generation at end | same file ✓ |

**Do not** try to rewrite all of `train_gpt2.c` in one shot. Change **one function** at a time and keep a smoke test (`make test_mla`).

In [ ]:
# Peek at llm.c attention if vendor is present (read-only — learning only)
path = ROOT / "vendor" / "llm.c" / "train_gpt2.c"
if path.exists():
    lines = path.read_text().splitlines()
    for i, line in enumerate(lines):
        if "void attention_forward" in line:
            start = max(0, i - 2)
            end = min(len(lines), i + 18)
            print("".join(f"{j+1:5d} | {lines[j]}\n" for j in range(start, end)))
            print("... compare to c/deepseek_v2/mla.c (latent KV path instead of Q,K,V in one buffer)")
            break
else:
    print("Clone vendor with: ./scripts/setup_vendor.sh")
    print("Then re-run this cell to see attention_forward() from llm.c.")


## C smoke test (piece 1 — MLA)

After you understand this notebook, open **`12.MLA.ipynb`**, then build the matching C:

```bash
cd c && make test_mla && ./bin/test_mla
```

The binary prints MLA vs MHA cache sizes — same numbers as the Python cell above.

In [ ]:
RUN_C = False  # set True to compile/run C smokes

# Optional: run the C MLA smoke test from the notebook (needs gcc + make)
import subprocess
import shutil

if shutil.which("make") and shutil.which("gcc"):
    r = subprocess.run(["make", "test_mla"], cwd=str(C_DIR), capture_output=True, text=True)
    print(r.stdout or r.stderr)
    if r.returncode != 0:
        print("make failed — run manually from the c/ directory")
else:
    print("Install gcc/make to run from notebook, or run: cd c && make test_mla")


## What to do next

1. **Notebook 12** — MLA (why `c_kv` is smaller than caching full K,V)
2. **Notebook 13** — MoE router + experts (brand new vs GPT-2)
3. **Notebooks 14–16** — assemble, train, sample
4. Read **`c/deepseek_v2/mla.c`** with **`llmc/deepseek_v2.py`** side by side

Skip “V1” unless you only want historical checkpoints; skip V4 until V2 feels obvious.